In [1]:
### 1. Load and Prepare Image
from pathlib import Path
import numpy as np
import tifffile as tiff

# Set your input path here
input_path = Path(r"C:\Users\adima\Desktop\Final_dataset\train\PN_KC_488_568_Brain5_CalyxL_LSMPlus_16bit.ome.tif")

def load_and_standardize(path):
    """Loads a 4D image and standardizes its dimension order to (Z, C, Y, X)."""
    img = tiff.imread(str(path))
    if img.ndim != 4:
        raise ValueError(f"Expected a 4D image, got shape {img.shape}")

    # Standardize to (Z, C, Y, X)
    if img.shape[0] == 2:          # (C, Z, Y, X) -> (Z, C, Y, X)
        img = np.transpose(img, (1, 0, 2, 3))
    elif img.shape[1] == 2:        # (Z, C, Y, X) - already correct
        pass
    else:
        raise ValueError(f"Cannot infer channel axis from shape {img.shape}")

    return img

# Load the full stack
img_stack = load_and_standardize(input_path)
print("Standardized stack shape:", img_stack.shape)

# --- Select Slice 32 ---
slice_index = 32
img_slice = img_stack[slice_index].copy() # Shape: (C, Y, X)
print(f"Selected slice {slice_index}. Shape: {img_slice.shape}")


Standardized stack shape: (38, 2, 1101, 1101)
Selected slice 32. Shape: (2, 1101, 1101)


In [2]:
### 2. Apply Rolling Ball Background Subtraction
from skimage.restoration import rolling_ball

print("Applying rolling ball background subtraction...")
img_bg_sub = np.zeros_like(img_slice, dtype=np.float32)
ball_radius = 50 # You can adjust this value

# Channel 0: Red, Channel 1: Green
for c, channel_name in enumerate(['Red', 'Green']):
    print(f"  - Processing {channel_name} channel...")
    background = rolling_ball(img_slice[c, :, :], radius=ball_radius)
    img_bg_sub[c, :, :] = img_slice[c, :, :] - background

print("Background subtraction complete.")


Applying rolling ball background subtraction...
  - Processing Red channel...
  - Processing Green channel...
Background subtraction complete.


In [12]:
### 3. Normalize Image for Segmentation
print("Normalizing image for segmentation...")

# Create an empty array for the normalized image
img_normalized = np.zeros_like(img_bg_sub, dtype=np.float32)

# Normalize each channel based on percentiles
for c, channel_name in enumerate(['Red', 'Green']):
    print(f"  - Normalizing {channel_name} channel...")
    channel_data = img_bg_sub[c]
    
    # Calculate percentiles
    p_low = np.percentile(channel_data, 0.0001)
    p_high = np.percentile(channel_data, 99.999)
    
    # Clip the data to the percentile range
    clipped_data = np.clip(channel_data, p_low, p_high)
    
    # Normalize to 0-1 range
    if p_high > p_low:
        img_normalized[c] = (clipped_data - p_low) / (p_high - p_low)
    else:
        # Handle the case where the channel is flat
        img_normalized[c] = np.zeros_like(channel_data, dtype=np.float32)

print("Normalization complete.")


Normalizing image for segmentation...
  - Normalizing Red channel...
  - Normalizing Green channel...
Normalization complete.


In [29]:
### 4. Watershed Segmentation with Peak-Finding Seeding
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

print("Performing watershed segmentation with peak-finding seeding...")

# --- Tunable Parameters ---
# Defines the brightness for an area to be considered for seeding (e.g., 90 = top 10% brightest pixels)
SEED_THRESHOLD_PERCENTILE = 90

# Defines the boundary for watershed growth. Lower values allow more growth.
BOUNDARY_THRESHOLD_PERCENTILE = 70

# Minimum number of pixels separating peaks. Prevents multiple seeds in one object.
MIN_DISTANCE = 10
# --------------------------

# Create empty arrays to store the results
watershed_labels = np.zeros_like(img_normalized, dtype=np.int32)
seed_masks = np.zeros_like(img_normalized, dtype=bool)

# Process each channel separately
for c, channel_name in enumerate(['Red', 'Green']):
    print(f"  - Segmenting {channel_name} channel...")
    
    image = img_normalized[c]
    
    # Initialize empty masks and labels for this channel
    labeled_seeds = np.zeros_like(image, dtype=np.int32)
    
    if np.any(image > 0):
        # 1. Define a high-intensity area to look for seeds
        seed_area_threshold = np.percentile(image[image > 0], SEED_THRESHOLD_PERCENTILE)
        seed_area_mask = image > seed_area_threshold

        # 2. Find distinct peaks ONLY within the high-intensity seed area.
        seed_coords = peak_local_max(
            image,
            min_distance=MIN_DISTANCE,
            labels=seed_area_mask
        )
        
        # 3. Create a marker image from the distinct seed coordinates
        markers = np.zeros_like(image, dtype=bool)
        if seed_coords.size > 0:
            markers[tuple(seed_coords.T)] = True
        
        # Label the final, clean seeds
        labeled_seeds, _ = ndi.label(markers)
        
        # Store the boolean seed mask for visualization
        seed_masks[c] = markers

        # 4. Run the watershed algorithm with a separate, stricter boundary
        boundary_threshold = np.percentile(image[image > 0], BOUNDARY_THRESHOLD_PERCENTILE)
        watershed_mask = image > boundary_threshold
        
        # The landscape is the distance from the center of the growth area
        distance = ndi.distance_transform_edt(watershed_mask)
        
        # Run watershed, constrained by the watershed_mask
        labels = watershed(-distance, labeled_seeds, mask=watershed_mask)
        watershed_labels[c] = labels

print("Watershed segmentation complete.")


Performing watershed segmentation with peak-finding seeding...
  - Segmenting Red channel...
  - Segmenting Green channel...
Watershed segmentation complete.


In [30]:
### 5. Visualization with Napari
import napari

viewer = napari.Viewer()

channel_colors = ['red', 'green']

for c, color in enumerate(channel_colors):
    # Add the normalized image, initially hidden
    viewer.add_image(
        img_normalized[c],
        name=f'Normalized - {color.capitalize()}',
        colormap=color,
        blending='additive',
        visible=False
    )
    
    # Add the seeds as a points layer, initially hidden
    seed_points = np.argwhere(seed_masks[c])
    viewer.add_points(
        seed_points,
        name=f'Seeds - {color.capitalize()}',
        face_color=color,
        size=5,
        visible=False
    )
    
    # Add the final watershed labels, visible by default
    viewer.add_labels(
        watershed_labels[c],
        name=f'Watershed Labels - {color.capitalize()}'
    )

napari.run()
